In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sqlite3
from datetime import datetime
from pathlib import Path
import pandas as pd
from pathlib import Path
import re

import arcgis
from arcgis.gis import GIS
import sys

# 1. Bepaal het absolute pad van de bovenliggende map (de project root)
project_root = str(Path.cwd().parent)

# 2. Voeg de project root toe aan sys.path, zodat Python de 'src' map kan vinden
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.utils as utils
import json

# import urllib3
# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


### Connect to AGOL

<div class="alert alert-block alert-info">This notebook uses the ArcGIS API for Python. For more information, see the <a href="https://developers.arcgis.com/python/">ArcGIS API for Python documentation and guides</a>.</div>

 The `create()` method is useful for batch creating blank surveys for which you can reassign ownership to another colleague to complete the design and publish. If you already have an XLSForm, you can use the `create()` and `publish()` methods to automate creating and publishing surveys.

 This sample notebook demonstrates how you can automate creating and publishing surveys using ArcGIS API for Python.

In [4]:
# Get credentials from key vault into environment
# Define your local specific paths 
LOCAL_DB = "G:\Mijn Drive\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"

# Run the function
AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(db_path=LOCAL_DB, entry_name=ENTRY_TITLE)

✅ Success: Credentials for 'AGOL' loaded into environment variables!


In [5]:
agol_url = "https://gisservices.inbo.be/portal"
gis = GIS(agol_url, AGOL_USER, AGOL_PASS)

In [6]:
survey_manager = arcgis.apps.survey123.SurveyManager(gis)

## Update existing survey - Might break AGOL Datastore

In [ ]:
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab1-4.xlsx")
survey_name = xlsform_path.stem  # Extract the survey name from the XLSForm file name 
surveys = survey_manager.surveys
    
# Safely find matches using the .get() method
matched_surveys = [s for s in surveys if s.properties.get('title') == survey_name]

if matched_surveys:
    # Return the ID of the first matching survey found
    print(matched_surveys[0].properties.get('id'))

# If nothing matches, gather available titles to provide a helpful error message
available_titles = [s.properties.get('title') for s in surveys if s.properties.get('title')]
print(f"Survey '{survey_name}' not found in the connected portal.\n")
print(f"Available surveys: {available_titles}")

# Instatiate surveymanager object
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
FIELD_MAP_ID = '64c1f0bd02344d5ebf41c3dd320615bc'

# Path to XLSForm file
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab1-4.xlsx")
survey_name = xlsform_path.stem 

# Fetch old survey ID by name
old_survey_id = utils.get_survey_id_by_name(survey_manager, survey_name)
print(f"Bestaande survey ({old_survey_id}) aan het updaten...")
    
# Haal de bestaande survey ops
mijn_survey = survey_manager.get(old_survey_id)

# Overschrijf het formulier met het nieuwe Excel-bestand
mijn_survey.publish(
    xlsform=str(xlsform_path),
    schema_changes=True,  # Sta schemawijzigingen toe
)
print("Survey succesvol overschreven!")

# Fetch object ID of new survey
new_survey_id = utils.get_survey_id_by_name(survey_manager, survey_name)
print(f"Nieuw survey object aangemaakt: {new_survey_id}")



c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

Connection to 172.31.1.204:9876 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
(Error Code: 500)


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

## Remove and Create a survey

You can create surveys using ArcGIS API for Python with the `create()` method. The `create()` method creates an empty form item and hosted feature service in the folder supplied to the method or a new folder created with the survey. The output of the `create()` method is a single <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.Survey">`Survey`</a> object. 

Surveys created by the `create()` method create the equivalent of a new unpublished web designer survey. After creating a survey with ArcGIS API for Python, if you don't have an XLSForm or existing survey design, you can use the <a href="https://doc.arcgis.com/en/survey123/browser/create-surveys/publishsurvey.htm">Survey123 web designer</a> to design and publish your survey. 

The first step is to connect to your GIS.

Next, a `SurveyManager` is defined, and a new survey is created. A survey in the Survey Manager is a single instance of a survey project that contains the item information and properties and provides access to the underlying survey dataset. For more information on Survey Manager, see the <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.SurveyManager" target="_blank">API Reference for the ArcGIS API for Python</a>.

To create a survey, use the `create()` method on a `SurveyManager` instance. 

The `create()` method only requires the `title` parameter. By default, the `create()` method will automatically create a folder using the `Survey-{title}` naming convention consistent with other Survey123 publishing apps. Alternatively, if you want to store your survey in a folder that already exists in your content, you can use the `folder` parameter and specify the name or folder ID.

The result of the `create()` method is a <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#survey" target="_blank">Survey</a> object. At this stage, the survey is in a draft state and has no schema or questions. You can use the Survey123 web designer to design your survey and publish from there, use the `publish()` method with an existing XLSForm design, or use a third-party Python package to design your survey directly in Python and call the `publish()` method. 

To publish a survey using the `publish()` method, an existing `Survey` object is required. The `Survey` can be an unpublished survey created by the `create()` method or an existing published survey in your content. It can also be an unpublished blank survey created with the Survey123 web designer (any designs will be overwritten by the `publish()` method's required XLSForm.). 

When publishing surveys with the `publish()` method, important considerations are as follows:

- Any survey is treated as a survey published with Survey123 Connect, which means you can no longer use the web designer to edit the portions of the survey that are managed by the XLSForm. For example, the survey title and survey questions cannot be edited. Themes, webhooks, and sharing options can still be edited in the web designer. 
- New surveys will create and use a `_form` view as the submission endpoint, which means you can no longer update the feature service schema in Survey123 Connect. 
- Existing surveys will maintain their submission endpoint.

A `schema_changes` parameter is available to modify the schema of your submission endpoint. For more information on the `schema_changes` parameter please see the <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.Survey.publish" target="_blank">ArcGIS API for Python</a> documentation.

In this example, the required `xlsform` parameter is set to the path where the XLSForm is located and optional parameters are set: 

- `info` is set to enable the Inbox.
- `enable_delete_protection` is enabled for the form item and its related content. 

##### Remove existing survey

In [8]:
# Instatiate surveymanager object
survey_manager = arcgis.apps.survey123.SurveyManager(gis)

# Path to XLSForm file
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab9.xlsx")
survey_name = xlsform_path.stem
print(f"Survey naam afgeleid van XLSForm: '{survey_name}'")
print(f" [*] Pad naar XLSForm: '{xlsform_path}'")

survey_name = 'survey_short_test'

Survey naam afgeleid van XLSForm: 'xlsform_hab9'
 [*] Pad naar XLSForm: 'Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab9.xlsx'


In [9]:
# 1. Haal de unieke ID van het hoofdformulier op via de naam
old_survey_id = utils.get_survey_id_by_name(survey_manager, survey_name)

if not old_survey_id:
    print(
        f" [!] Geen survey gevonden met de naam '{survey_name}'. Annuleren."
    )
else:
    print(f" [*] Bestaande survey ({old_survey_id}) aan het updaten...")

form_item = gis.content.get(old_survey_id)
if not form_item:
    print(f" [!] Kon het item met ID {old_survey_id} niet ophalen.")

# Sla de unieke formulier-ID op (Survey123 gebruikt deze ID vaak in de servicenamen)
form_id = form_item.id
user = gis.users.get(form_item.owner)

# 2. Zoek de map waarin deze specifieke survey leeft
folder_name, folder_id = utils.browse_folders_for_survey(gis, old_survey_id)
print(f"Survey bevindt zich in map: '{folder_name}' (ID: {folder_id})")

c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.inbo.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

 [*] Bestaande survey (79c832d6a19349e38df1c34c45f3fadf) aan het updaten...
Survey bevindt zich in map: 'Survey-LSVI App Test Auto' (ID: 5ddb1d7494d34125a4f6a7047478810e)


In [10]:
# 3. Verzamel ALLE items die bij deze survey horen in één platte lijst
# Haal ALLE items op uit deze map
all_folder_items = user.items(folder=folder_name)
items_to_delete = []

for item in all_folder_items:
    is_target_item = (
        item.id == form_id or 
        item.title.lower().startswith(survey_name.lower()) or 
        form_id in item.title or
        (item.name and form_id in item.name)
    )
    if is_target_item:
        items_to_delete.append(item)

if not items_to_delete:
    print(" -> Geen gekoppelde elementen gevonden om te verwijderen.")

In [11]:
items_to_delete

[<Item title:"survey_short_test" type:Form owner:joost.neujens@inbo.be>,
 <Item title:"survey_short_test" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"survey_short_test_form" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"survey_short_test" type:Web Map owner:joost.neujens@inbo.be>]

In [12]:
items_to_delete.sort(key=utils.get_delete_priority)

In [13]:
# items_to_delete
for item in items_to_delete:
    try:
        item.protect(enable=False)
    except Exception:
        pass

In [14]:
pass_number = 1
max_passes = 1  # Meestal is alles in 2 passes al volledig weg

while items_to_delete and pass_number <= max_passes:
    print(f"\n--- Start Verwijderronde {pass_number} ---")
    leftover_items = []
    deleted_any_this_pass = False

    for item in items_to_delete:
        try:
            # Poging tot verwijderen
            item.delete()
            print(f" [✓] Succesvol verwijderd: {item.title} ({item.type})")
            deleted_any_this_pass = True
        except Exception as e:
            # Als het faalt (bijv. Error 400 wegens gerelateerd item), bewaren we hem voor de volgende ronde
            leftover_items.append(item)

    # Update de hoofdlijst met de items die we moesten skippen
    items_to_delete = leftover_items

    # Veiligheidsklep: als een hele ronde niks heeft kunnen verwijderen,
    # zitten we vast op een échte harde fout (bijv. rechten) en moeten we stoppen om infinite loops te voorkomen.
    if not deleted_any_this_pass and items_to_delete:
        print("\n [!] Systeem zit vast: resterende items hebben permanente blokkades.")
        break

    pass_number += 1


--- Start Verwijderronde 1 ---
 [✓] Succesvol verwijderd: survey_short_test (Web Map)
 [✓] Succesvol verwijderd: survey_short_test_form (Feature Service)
 [✓] Succesvol verwijderd: survey_short_test (Feature Service)
 [✓] Succesvol verwijderd: survey_short_test (Form)


##### Recreate survey

In [164]:
survey_manager = arcgis.apps.survey123.SurveyManager(gis)

In [165]:
# xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\survey_short_test.xlsx")
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab9.xlsx")
survey_name = xlsform_path.stem  # Extract the survey name from the XLSForm file name 


In [166]:
target_folder = "Survey-LSVI App Test Auto"
thumbnail_path = r"Q:\Projects\PRJ_GIS\lsvi-app-testing\inbo_logo.jpg"

In [167]:
print(xlsform_path)
print(survey_name)
survey_title = survey_name

Q:\Projects\PRJ_GIS\lsvi-app-testing\output\xlsform_hab9.xlsx
xlsform_hab9


In [110]:
# Check if folder already exists
user = gis.users.me
bestaande_mappen = [folder.name for folder in user.folders]

if target_folder not in bestaande_mappen:
    print(f"Map '{target_folder}' bestaat nog niet. Aanmaken...")
    try:
        gis.content.create_folder(folder=target_folder)
        print(f" [✓] Map '{target_folder}' succesvol aangemaakt.")
    except Exception as e:
        print(
            f" [!] Waarschuwing bij het aanmaken van map '{target_folder}': {e}"
        )

In [111]:
valid_thumbnail = (
        thumbnail_path if Path(thumbnail_path).exists() else None
    )

In [112]:
print(f" -> Concept aanmaken in map '{target_folder}'...")
new_survey = survey_manager.create(
    title=str(survey_title),
    folder=target_folder,
    tags="LSVI, Survey123, Python",
    summary=f"Automatisch gegenereerde LSVI survey op basis van {xlsform_path.name}.",
    description=f"Deze survey is automatisch gepubliceerd via Python vanuit het bestand: {xlsform_path.name}.",
    thumbnail=valid_thumbnail,
)

# print(f" -> Publiceren naar ArcGIS Online begonnen (dit kan even duren)...")
# new_survey.publish(
#     xlsform=str(xlsform_path),
#     # enable_delete_protection=False,  # Zorgt dat je delete-script de tabel kan overschrijven
#     enable_sync=True,  # Cruciaal voor offline gebruik in de Survey123 veld-app
#     # thumbnail=valid_thumbnail,
#     schema_changes=False, # Joost Given we remove and reupload, better to set to False?
# )

 -> Concept aanmaken in map 'Survey-LSVI App Test Auto'...


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory 'C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmp69dh1zwp'>
  _warnings.warn(warn_message, ResourceWarning)


In [113]:
str(xlsform_path)

'Q:\\Projects\\PRJ_GIS\\lsvi-app-testing\\output\\xlsform_hab1-4.xlsx'

In [114]:
published_survey = new_survey.publish(
    xlsform=str(xlsform_path),
    info=
        {"queryInfo": {
            "mode": "manual",
            "editEnabled": True,
            "copyEnabled": False
            }
        }, 
    enable_delete_protection=False
    )

published_survey

c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\arcgis\apps\survey123\_survey.py:1613: ResourceWarning: unclosed file <_io.BufferedReader name='C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmpkk04vbui\\55460219a4604014a00440ddb7e2f3ec\\esriinfo\\xlsform_hab1-4.xlsx'>
  xform = _xls2xform(xlsform)
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory 'C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmpg563252k'>
  _warnings.warn(warn_message, ResourceWarning)
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory 'C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmpylhaii72'>
  _warnings.warn(warn_message, ResourceWarning)


<Survey @ xlsform_hab1-4>

In [ ]:
# C. Haal de nieuwe unieke Item ID op ter verificatie
new_survey_id = utils.get_survey_id_by_name(gis, survey_title)
print(f" [✓] Succesvol gepubliceerd! Item ID: {new_survey_id}")

## Update BWK Field Map (Web Map)

In [159]:
FIELD_MAP_ID = "ad1a1d268ddf4f02b1bb48f6f1b85f1c"

In [160]:
# Get the web map's internal data configuration
map_item = gis.content.get(FIELD_MAP_ID)
map_data = map_item.get_data()

# Convert configuration to a string to easily replace the old ID globally
map_data_str = json.dumps(map_data)

In [161]:
map_data_str

'{"operationalLayers": [{"id": "Veld_BWKhab2022_8593", "opacity": 1, "title": "SBZ-V", "url": "https://gisservices.inbo.be/arcgis/rest/services/Veld/Veld_BWKhab2026/FeatureServer/11", "visibility": false, "itemId": "e7b18684c7c94bb4a46a515ef73186bc", "layerType": "ArcGISFeatureLayer", "layerDefinition": {"capabilities": "Query,Sync"}, "capabilities": "Query,Sync", "popupInfo": {"popupElements": [{"type": "fields"}, {"type": "attachments", "displayType": "auto"}], "showAttachments": true, "fieldInfos": [{"fieldName": "VOLGNR", "format": {"digitSeparator": true, "places": 0}, "isEditable": true, "label": "VOLGNR", "tooltip": "", "visible": true}, {"fieldName": "GEBCODE", "isEditable": true, "label": "GEBCODE", "tooltip": "", "visible": true}, {"fieldName": "NA2000CODE", "isEditable": true, "label": "NA2000CODE", "tooltip": "", "visible": true}, {"fieldName": "GEBNAAM", "isEditable": true, "label": "GEBNAAM", "tooltip": "", "visible": true}, {"fieldName": "BEHOUD", "isEditable": true, "la

In [ ]:


# Scenario A: The survey ID did not change (e.g., standard overwrite publish)
if old_survey_id == new_survey_id:
    print("ℹ️ The survey ID did not change. No web map update is required.")

# Scenario B: The survey ID changed (e.g., wipe and recreate workflow)
elif old_survey_id in map_data_str:
    # Safely swap out ONLY the old target ID with the new one
    updated_map_data_str = map_data_str.replace(old_survey_id, new_survey_id)
    updated_map_data = json.loads(updated_map_data_str)
    
    # Push the updated data config back to the Web Map item in AGOL
    map_item.update(data=updated_map_data)
    print(f"Success! Updated the pop-up link in the Web Map from {old_survey_id} to {new_survey_id}.")

# Scenario C: Safety net in case the survey ID isn't found in the popup at all
else:
    print(
        f"⚠️ Warning: The old survey ID ({old_survey_id}) for '{survey_name}' "
        "was not found in the Web Map pop-up configuration. No update was made."
    )

## Share survey items with correct group

In [15]:
GROUP_NAME = "LSVItesters"
TARGET_FOLDER = "Survey-LSVI App Test Auto"

In [ ]:
# Get group
lsvi_group = gis.groups.search(query=f"title: {GROUP_NAME}", max_groups=15)
lsvi_group = lsvi_group[0]


lsvi_group

<Group title:"LSVItesters" owner:Carine.WILS@INBO.BE>

In [22]:
# Get group id
lsvi_group_id = lsvi_group.id

# Get list of items
group_items = lsvi_group.items()

# Get all items in target folder
user = gis.users.me

# Share with group
folder_items = list(user.items(folder=TARGET_FOLDER))
print(f"Deel {len(folder_items)} items uit map '{TARGET_FOLDER}' met groep '{lsvi_group.title}'...")


Deel 13 items uit map 'Survey-LSVI App Test Auto' met groep 'LSVItesters'...


In [23]:
folder_items

[<Item title:"BWK habitat - survey123 Test (Joost)" type:Web Map owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab1-4" type:Form owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab1-4" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab1-4_form" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab1-4" type:Web Map owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab5-7" type:Form owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab5-7" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab5-7_form" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab5-7" type:Web Map owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab9" type:Form owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab9" type:Feature Layer Collection owner:joost.neujens@inbo.be>,
 <Item title:"xlsform_hab9_form" type:Feature Layer Collection owner:joost.neujens@inbo.be>,


In [24]:


for item in folder_items:
    try:
        # item.share() deelt het item direct met de opgegeven groep(en)
        item.share(groups=[lsvi_group])
        print(f" [✓] Gedeeld: {item.title} ({item.type})")
    except Exception as e:
        print(f" [!] Fout bij delen van {item.title}: {e}")

c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


 [✓] Gedeeld: BWK habitat - survey123 Test (Joost) (Web Map)
 [✓] Gedeeld: xlsform_hab1-4 (Form)


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


 [✓] Gedeeld: xlsform_hab1-4 (Feature Service)


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


...

c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


 [✓] Gedeeld: xlsform_hab9 (Feature Service)


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


 [✓] Gedeeld: xlsform_hab9_form (Feature Service)
 [✓] Gedeeld: xlsform_hab9 (Web Map)


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
